# Go2 Track 2 Bonus Project: Colab Starter Notebook

Use this notebook to set up the repo, inspect the interfaces, optionally train or reuse a low-level checkpoint, run single-policy track evaluation, and prepare submission metadata.

It starts from a weak baseline. Your leaderboard submission should train a learned high-level planner for the fixed 5D -> [vx, vy, yaw_rate] interface.


## 1. Configure repository URLs

Leave the default repo URL unless you are working from your own fork. If you rerun setup after editing files, keep `RESET_COURSE_REPO = False` so your local changes are not deleted.


In [1]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

COURSE_REPO_URL = "https://github.com/aooooofu-code/Final-Project-Track-2-Bonus-Project.git"
COURSE_REPO_BRANCH = "main"
TEAM_NAME = "change_me"
RESET_COURSE_REPO = False
BASE_DIR = Path("/content") if Path("/content").exists() else Path("/tmp/go2_track_bonus_colab")
BASE_DIR.mkdir(parents=True, exist_ok=True)
COURSE_REPO_DIR = BASE_DIR / "go2_track_bonus_repo"

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"

PLAYGROUND_DIR = BASE_DIR / "mujoco_playground"
UNITREE_DIR = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        if RESET_COURSE_REPO:
            print(f"+ remove existing course repo at {target_dir}")
            shutil.rmtree(target_dir)
        else:
            print(f"+ reuse existing course repo at {target_dir}")
            return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")


Running inside Colab.


## 2. Install system packages and clone repositories


In [2]:
import shutil
if shutil.which("ffmpeg") is None:
    if Path("/content").exists():
        run(["apt-get", "update", "-qq"])
        run(["apt-get", "install", "-y", "ffmpeg"])
    else:
        print("[local] ffmpeg is not on PATH; imageio-ffmpeg from requirements is enough for local tests.")
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground || true

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}
%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .
%cd {COURSE_REPO_DIR}

import sys
if str(PLAYGROUND_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_DIR.resolve()))

import jax
import mujoco_playground

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())
print("mujoco_playground imported from:", mujoco_playground.__file__)
expected_playground = str(PLAYGROUND_DIR.resolve())
if expected_playground not in str(Path(mujoco_playground.__file__).resolve()):
    raise RuntimeError(f"Expected mujoco_playground to be imported from {expected_playground}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
+ git clone https://github.com/google-deepmind/mujoco_playground.git /content/mujoco_playground
+ git -C /content/mujoco_playground fetch --all --tags
+ git -C /content/mujoco_playground checkout dd38c285c6d54266287081e516109f0b15985818
+ git clone https://github.com/unitreerobotics/unitree_mujoco.git /content/unitree_mujoco
+ git -C /content/unitree_mujoco fetch --all --tags
+ git -C /content/unitree_mujoco checkout 1a37b051a10be723405b7ed6dc839361af036d88
+ git clone https://github.com/aooooofu-code/Final-P

## 3. Read the assignment requirements

Skim the short specs before editing.


In [3]:
%cd {COURSE_REPO_DIR}
!sed -n '1,180p' docs/assignment_requirements.md
!sed -n '1,140p' docs/controller_interface.md
!sed -n '1,120p' docs/high_level_optimization_guide.md


/content/go2_track_bonus_repo
# Track 2 Requirements

Goal: run Go2 as far as possible around a 200 m oval track in MuJoCo.

## Options

- Proposal-based final project.
- Go2 oval-track leaderboard route.
- Both, for bonus.

## Leaderboard Route

```text
5D track observation -> [vx, vy, yaw_rate] -> Go2 low-level policy
```

Use `docs/controller_interface.md`. The low-level checkpoint should stay
compatible with the HW1 Brax PPO format.
This repo evaluates one submission at a time; ranking compares submitted
outputs.

Leaderboard submissions must train a learned high-level planner for this fixed
interface. The provided starter planner is only a weak baseline for debugging.
Keep the 5D input and 3D output fixed; change the planner internals. The
official scene is fixed: 200 m centerline, 18.25 m turn radius, and 2.0 m half
width. Do not change the track geometry to improve score.

## Allowed

- Reuse a HW1 checkpoint.
- Retrain or modify the low-level Go2 policy.
- Train a learned high-

## 4. Copy Go2 assets


In [4]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}


/content/go2_track_bonus_repo
Copied 16 assets into /content/go2_track_bonus_repo/go2_pg_env/xmls/assets


## 5. Inspect the low-level Go2 environment


In [5]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_2


/content/go2_track_bonus_repo
{
  "environment_name": "Go2JoystickFlatTerrain",
  "stage_name": "stage_2",
  "backend_impl": "jax",
  "control_dt": 0.02,
  "sim_dt": 0.004,
  "episode_length": 1000,
  "action_size": 12,
  "actor_obs_size": 48,
  "critic_obs_size": 123,
  "observation_layout": {
    "state": [
      [
        "local_linvel",
        3
      ],
      [
        "gyro",
        3
      ],
      [
        "gravity",
        3
      ],
      [
        "joint_pos_error",
        12
      ],
      [
        "joint_vel",
        12
      ],
      [
        "last_action",
        12
      ],
      [
        "command",
        3
      ]
    ],
    "privileged_state_extra": [
      [
        "gyro_clean",
        3
      ],
      [
        "accelerometer",
        3
      ],
      [
        "gravity_clean",
        3
      ],
      [
        "local_linvel_clean",
        3
      ],
      [
        "global_angvel",
        3
      ],
      [
        "joint_pos_error_clean",
       

## 6. Read the important starter files


In [6]:
!sed -n '1,180p' go2_pg_env/joystick.py
!sed -n '1,220p' go2_pg_env/track.py
!sed -n '1,220p' track_bonus/controller_interface.py
!sed -n '1,220p' track_bonus/planner.py
!sed -n '1,220p' run_track_bonus.py


"""Joystick locomotion task for the local Go2 environment.

This task is adapted from MuJoCo Playground's Go1 joystick task. The local
changes are intentionally small so that students can compare the official
baseline against a course-specific Go2 variant.

Observation summary
-------------------
state (actor input):
    [local_linvel(3), gyro(3), gravity(3),
     joint_pos_error(12), joint_vel(12),
     last_action(12), command(3)]  -> 48 dims

privileged_state (critic-only input during training):
    state + extra simulator-only signals -> 123 dims

Action summary
--------------
The policy outputs 12 joint offsets. The final motor target is:
    target_joint_pos = default_pose + action_scale * policy_action
"""

from __future__ import annotations

from typing import Any, Dict, Optional, Union

import jax
import jax.numpy as jp
from ml_collections import config_dict
from mujoco import mjx
from mujoco.mjx._src import math
import numpy as np

from mujoco_playground._src import mjx_env



## 7. Define a Colab-friendly low-level training config

This is a normal Colab training starting point, not a quick test. Reduce the step counts for experiments if needed.


In [7]:
import json

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)


wrote /content/go2_track_bonus_repo/configs/colab_runtime_config.json


## 8. Dry-run training config


In [8]:
!python train.py --config configs/colab_runtime_config.json --dry-run


{
  "homework_name": "Final Project Track 2 Bonus: Go2 Low-Level Locomotion Baseline",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "framework": "MuJoCo Playground + Brax PPO + MJX",
  "backend_impl": "jax",
  "actor_obs_key": "state",
  "critic_obs_key": "privileged_state",
  "use_domain_randomization": true,
  "seed": 0,
  "control": {
    "ctrl_dt": 0.02,
    "sim_dt": 0.004,
    "action_scale": 0.5,
    "action_type": "absolute_joint_position_target",
    "torque_mapping": "position_target_through_pd_actuator"
  },
  "course_budget": {
    "baseline_total_env_steps": 15000000,
    "leaderboard_max_env_steps": 30000000,
    "flat_terrain_only": true,
    "require_colab_gpu_runtime": true
  },
  "training_defaults": {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [
      256,
      256,
      128
    ],
    "value_hidden_layer_sizes": [
      256,
      256,
      128
    ]
  },
  "st

## 9. Train or reuse a low-level checkpoint

Use a newly trained checkpoint or point `CHECKPOINT_DIR` to a HW1 `best_checkpoint`.


In [18]:
# Uncomment the command below to train a new low-level policy.
# !python train.py \
#   --config configs/colab_runtime_config.json \
#   --stage both \
#   --output-dir artifacts/low_level_train

# Or set CHECKPOINT_DIR to an existing HW1 best_checkpoint, for example:
# CHECKPOINT_DIR = Path("/content/path/to/your/HW1/best_checkpoint")
CHECKPOINT_DIR = Path(
    "/content/go2_course_repo/artifacts/run_ours/best_checkpoint"
)
PLANNER_CONFIG = COURSE_REPO_DIR / "configs" / "strong_planner.json"
print("checkpoint path:", CHECKPOINT_DIR)
print("planner config:", PLANNER_CONFIG)
print("checkpoint exists:", CHECKPOINT_DIR.exists())


checkpoint path: /content/go2_course_repo/artifacts/run_ours/best_checkpoint
planner config: /content/go2_track_bonus_repo/configs/strong_planner.json
checkpoint exists: True


In [10]:
!ls /content
!ls /content/go2_course_repo/artifacts/run_ours/best_checkpoint

go2_track_bonus_repo  mujoco_playground  sample_data  unitree_mujoco
ls: cannot access '/content/go2_course_repo/artifacts/run_ours/best_checkpoint': No such file or directory


In [11]:
%cd /content

!git clone https://github.com/aooooofu-code/EEC289A_Robotics-Homework.git go2_course_repo

/content
Cloning into 'go2_course_repo'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 113 (delta 28), reused 22 (delta 22), pack-reused 68 (from 2)
Receiving objects: 100% (113/113), 6.79 MiB | 41.89 MiB/s, done.
Resolving deltas: 100% (38/38), done.


In [12]:
!ls /content/go2_course_repo/artifacts/run_ours/best_checkpoint

array_metadatas       manifest.json   ocdbt.process_0
_CHECKPOINT_METADATA  manifest.ocdbt  ppo_network_config.json
d		      _METADATA       _sharding


In [15]:
%cd /content/go2_track_bonus_repo

/content/go2_track_bonus_repo


## 10. Run single-policy track evaluation

The smoke command is short and writes to `track_eval_smoke`. For submission, run the full command into `track_eval`.


In [28]:
TRACK_EVAL_SMOKE_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval_smoke"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

if CHECKPOINT_DIR.exists():
    print("Found checkpoint. Running short no-render track smoke eval...")
    !python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config {PLANNER_CONFIG} \
      --config configs/colab_runtime_config.json \
      --output-dir {TRACK_EVAL_SMOKE_DIR} \
      --entry-name {TEAM_NAME} \
      --duration-seconds 120 \
      --no-render
else:
    print("Skipping track eval: CHECKPOINT_DIR does not exist yet.")
    print("Train a policy above or set CHECKPOINT_DIR to your HW1 best_checkpoint, then rerun this cell.")

# Full evaluation with video, after the smoke run works:
# !python run_track_bonus.py \
#   --checkpoint-dir {CHECKPOINT_DIR} \
#   --planner-config {PLANNER_CONFIG} \
#   --config configs/colab_runtime_config.json \
#   --output-dir {TRACK_EVAL_DIR} \
#   --entry-name {TEAM_NAME} \
#   --render-every 10 \
#   --render-fps 5


Found checkpoint. Running short no-render track smoke eval...
{
  "output_dir": "/content/go2_track_bonus_repo/artifacts/track_eval_smoke",
  "metrics": {
    "lap_completion": 0.46262512281409,
    "valid_distance_m": 92.525024562818,
    "finish_time": null,
    "mean_progress_speed": 0.7710418713568167,
    "alive_time": 120.0,
    "fall": false,
    "fall_step": null,
    "boundary_violation": false,
    "boundary_violation_step": null,
    "rms_lateral_error": 0.09446901164784317,
    "max_lateral_error": 0.22811126708984375,
    "min_boundary_margin_m": 1.7718887329101562,
    "energy_proxy": 7.620373249053955,
    "foot_slip_proxy": 0.8182185292243958,
    "total_time": 120.0,
    "num_steps": 6000
  },
  "scores": {
    "completion_score": 0.46262512281409,
    "speed_score": 0.8280558284757555,
    "line_keeping_score": 1.0,
    "stability_score": 1.0,
    "efficiency_score": 0.6,
    "composite_score": 0.7037924709614917
  }
}


In [ ]:
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

if CHECKPOINT_DIR.exists():
    print("Found checkpoint. Running full no-render track eval...")
    !python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config {PLANNER_CONFIG} \
      --config configs/colab_runtime_config.json \
      --output-dir {TRACK_EVAL_DIR} \
      --entry-name {TEAM_NAME} \
      --no-render
else:
    print("Skipping track eval: CHECKPOINT_DIR does not exist yet.")

Found checkpoint. Running full no-render track eval...


In [17]:
import json
from pathlib import Path

strong_config = {
    "planner_type": "starter_pd",
    "speed_mps": 0.75,
    "min_speed_mps": 0.20,
    "max_lateral_speed_mps": 0.18,
    "max_yaw_rate_radps": 0.75,
    "k_heading": 1.25,
    "k_lateral": 0.22,
    "heading_slowdown": 0.65,
    "stand_seconds": 0.2
}

path = Path("/content/go2_track_bonus_repo/configs/strong_planner.json")
path.write_text(json.dumps(strong_config, indent=2))
print(path)
print(path.read_text())

/content/go2_track_bonus_repo/configs/strong_planner.json
{
  "planner_type": "starter_pd",
  "speed_mps": 0.75,
  "min_speed_mps": 0.2,
  "max_lateral_speed_mps": 0.18,
  "max_yaw_rate_radps": 0.75,
  "k_heading": 1.25,
  "k_lateral": 0.22,
  "heading_slowdown": 0.65,
  "stand_seconds": 0.2
}


## 11. Train your high-level planner

You are expected to train a learned high-level planner. The command below is only a starter parameter search for debugging the loop; replace the planner internals with your own MLP, RL policy, or other trained policy while keeping the same 5D -> [vx, vy, yaw_rate] interface.

Do not change the track geometry. The evaluator uses the fixed official oval for reset, scoring, and rendering.

A valid learned planner has trained parameters, such as MLP weights. Store those weights in your submission and load them from `StarterTrackPlanner.load(planner_config)`.


In [ ]:
HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "highlevel_train"

# Starter option: black-box search over configs/starter_planner.json.
# !python train_highlevel_starter.py \
#   --checkpoint-dir {CHECKPOINT_DIR} \
#   --config configs/colab_runtime_config.json \
#   --output-dir {HIGHLEVEL_DIR} \
#   --iterations 8 \
#   --population 12

# After search, switch to the best config and rerun Step 10 full evaluation.
# For a learned planner, edit track_bonus/planner.py and keep StarterTrackPlanner.load(...).command(track_observation, t).
# PLANNER_CONFIG = HIGHLEVEL_DIR / "best_planner_config.json"


## 12. Create submission metadata


In [ ]:
import json
submission = {
    "team_name": TEAM_NAME,
    "track2_option": "leaderboard",
    "checkpoint_dir": "best_checkpoint",
    "planner_config": "planner_config.json",
    "planner_code": "track_bonus/planner.py",
    "planner_weights": "planner_weights.npz, if used",
    "high_level_planner_type": "learned",
    "track_eval": "track_eval/results.json",
    "notes": "Briefly describe your low-level training, learned high-level planner, and failed ideas."
}
(COURSE_REPO_DIR / "submission.json").write_text(json.dumps(submission, indent=2))
print((COURSE_REPO_DIR / "submission.json").read_text())


## 13. Final local checklist

This only checks local paths. If `track_eval/results.json` is missing, run the full evaluation command in Step 10.


In [ ]:
from pathlib import Path
expected = {
    "checkpoint": CHECKPOINT_DIR,
    "planner_config": PLANNER_CONFIG,
    "submission_json": COURSE_REPO_DIR / "submission.json",
    "track_eval_results": TRACK_EVAL_DIR / "results.json",
}
for label, path in expected.items():
    print(label, path, "OK" if path.exists() else "MISSING")
print("Submit best_checkpoint/, planner_config.json, planner weights if used, changed planner code if any, submission.json, track_eval/results.json, and your short report.")
